# FallEnterprise · LoRA Fine-Tune Template

Fine-tune Llama 3.1 8B on your business data using LoRA (Low-Rank Adaptation).

**What this notebook does:**
1. Loads Llama 3.1 8B base model in 4-bit quantized form
2. Loads your business data (JSONL format · one conversation per line)
3. Trains a LoRA adapter (small · trainable · fast)
4. Merges the adapter into the base model
5. Exports to GGUF format for Ollama

**Requirements:**
- NVIDIA GPU with ≥16GB VRAM (A100 / A6000 / RTX 4090 all fine)
- ~50GB free disk
- Hugging Face account with Llama 3.1 access granted
- HF token exported: `export HF_TOKEN=hf_...`

**Runtime:** 2-8 hours depending on dataset size and GPU

---

## Cell 1 · Install dependencies

In [ ]:
%pip install -q --upgrade transformers==4.44.2 datasets==3.0.1 peft==0.13.2 \
  trl==0.11.4 accelerate==0.34.2 bitsandbytes==0.44.1 sentencepiece==0.2.0 \
  huggingface_hub==0.25.2 wandb==0.18.3

## Cell 2 · Config

Edit these for your business.

In [ ]:
import os

BASE_MODEL      = 'meta-llama/Meta-Llama-3.1-8B-Instruct'

# ─── CHANGE THIS TO YOUR BUSINESS ────────────────────────────────
BUSINESS_SLUG   = 'yourbiz'                       # used in output paths
TRAIN_DATA_PATH = 'data/train.jsonl'              # your training conversations
EVAL_DATA_PATH  = 'data/eval.jsonl'               # 10-15% of data held out

# ─── LoRA hyperparams (defaults tuned for typical SMB dataset ~10-50k examples) ──
LORA_R          = 16     # rank · higher = more expressive, more compute
LORA_ALPHA      = 32     # scaling · usually 2× rank
LORA_DROPOUT    = 0.05
TARGET_MODULES  = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

# ─── training hyperparams ─────────────────────────────────────────
OUTPUT_DIR      = f'./lora-out-{BUSINESS_SLUG}'
EPOCHS          = 3
BATCH_SIZE      = 4               # per-device · lower if OOM
GRAD_ACCUM      = 4               # effective batch = BATCH_SIZE * GRAD_ACCUM = 16
LEARNING_RATE   = 2e-4
MAX_SEQ_LEN     = 2048

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Config loaded. Output → {OUTPUT_DIR}')

## Cell 3 · Data format

Your `train.jsonl` and `eval.jsonl` should have one JSON object per line, with a `messages` list:

```json
{"messages": [
  {"role": "system", "content": "You are a customer service agent for <BUSINESS>."},
  {"role": "user", "content": "Do you deliver on Sundays?"},
  {"role": "assistant", "content": "Yes — Sunday deliveries run 10am-3pm across the North Kent area. Anywhere else needs 48h notice. Want me to schedule one?"}
]}
```

**Sources of good training data:**
- Past customer emails (your best responses)
- Sales call transcripts (annotated with what worked)
- Support ticket resolutions
- Internal SOPs and playbooks
- FAQ documents

**Data quantity:**
- 500 examples: model learns your tone
- 5,000 examples: model learns your process
- 50,000 examples: model learns your business

**Data quality > quantity.** 500 excellent examples beat 5,000 mediocre ones.

In [ ]:
from datasets import load_dataset

train_ds = load_dataset('json', data_files=TRAIN_DATA_PATH, split='train')
eval_ds  = load_dataset('json', data_files=EVAL_DATA_PATH,  split='train')
print(f'train: {len(train_ds)}   eval: {len(eval_ds)}')
print('sample:', train_ds[0])

## Cell 4 · Load base model (4-bit quantized)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    attn_implementation='flash_attention_2' if torch.cuda.is_available() else 'eager',
)
model = prepare_model_for_kbit_training(model)
print(f'loaded {BASE_MODEL} · GPU mem used: {torch.cuda.memory_allocated()/1e9:.1f}GB')

## Cell 5 · Attach LoRA adapter

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Cell 6 · Format data with chat template

In [ ]:
def format_example(ex):
    return {
        'text': tokenizer.apply_chat_template(
            ex['messages'],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

train_ds = train_ds.map(format_example, remove_columns=train_ds.column_names)
eval_ds  = eval_ds.map(format_example, remove_columns=eval_ds.column_names)
print(train_ds[0]['text'][:400])

## Cell 7 · Train

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    bf16=True,
    logging_steps=10,
    save_strategy='epoch',
    eval_strategy='epoch',
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field='text',
    optim='paged_adamw_8bit',
    report_to='none',                 # set to 'wandb' if you want tracking
    save_total_limit=2,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=training_args,
)

trainer.train()
trainer.save_model(f'{OUTPUT_DIR}/final')

## Cell 8 · Merge LoRA into base + save merged model

In [ ]:
from peft import PeftModel

# reload base in fp16 (not 4-bit) so we can merge
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map='auto',
)
merged = PeftModel.from_pretrained(base, f'{OUTPUT_DIR}/final')
merged = merged.merge_and_unload()
merged.save_pretrained(f'{OUTPUT_DIR}/merged', safe_serialization=True)
tokenizer.save_pretrained(f'{OUTPUT_DIR}/merged')
print(f'merged model saved to {OUTPUT_DIR}/merged')

## Cell 9 · Convert to GGUF for Ollama

This uses llama.cpp's converter. Run this cell in a fresh terminal (not in the notebook) if you don't have llama.cpp installed.

In [ ]:
# clone llama.cpp if you don't have it
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git ../llama.cpp 2>/dev/null || echo 'llama.cpp exists'
!pip install -q -r ../llama.cpp/requirements.txt

# convert HF → GGUF (fp16)
!python ../llama.cpp/convert_hf_to_gguf.py {OUTPUT_DIR}/merged \
  --outfile {OUTPUT_DIR}/{BUSINESS_SLUG}-fp16.gguf --outtype f16

# quantize to Q4_K_M (good balance of size/quality for inference)
!../llama.cpp/build/bin/llama-quantize \
  {OUTPUT_DIR}/{BUSINESS_SLUG}-fp16.gguf \
  {OUTPUT_DIR}/{BUSINESS_SLUG}-Q4_K_M.gguf Q4_K_M

print(f'GGUF saved · load into Ollama with the Modelfile below')

## Cell 10 · Load into Ollama

Write a Modelfile pointing at your GGUF, then create the Ollama model.

In [ ]:
modelfile = f'''FROM {OUTPUT_DIR}/{BUSINESS_SLUG}-Q4_K_M.gguf

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER num_ctx 4096

TEMPLATE """<|begin_of_text|><|start_header_id|>{{{{ .Role }}}}<|end_header_id|>

{{{{ .Content }}}}<|eot_id|>"""

SYSTEM """You are the AI assistant for {BUSINESS_SLUG}, trained on our business data. Respond in the voice, values, and process of the business."""
'''

with open(f'{OUTPUT_DIR}/Modelfile', 'w') as f:
    f.write(modelfile)

print(f'Modelfile written. To load into Ollama:')
print(f'  docker exec fe_ollama ollama create fe-{BUSINESS_SLUG} -f {OUTPUT_DIR}/Modelfile')
print(f'Then add to docker/litellm.config.yaml:')
print(f'  - model_name: fe-{BUSINESS_SLUG}')
print(f'    litellm_params:')
print(f'      model: ollama/fe-{BUSINESS_SLUG}:latest')
print(f'      api_base: http://ollama:11434')

## Cell 11 · Eval harness (business-specific)

Run your held-out eval set through both the base model and the fine-tuned model.
Log win-rate by human review or LLM-judge.

In [ ]:
import json
from transformers import pipeline

pipe = pipeline('text-generation', model=f'{OUTPUT_DIR}/merged', tokenizer=tokenizer, device_map='auto')

eval_examples = list(load_dataset('json', data_files=EVAL_DATA_PATH, split='train'))
results = []
for i, ex in enumerate(eval_examples[:50]):        # first 50 for a quick pass
    prompt = tokenizer.apply_chat_template(
        ex['messages'][:-1],                      # everything except the target response
        tokenize=False,
        add_generation_prompt=True,
    )
    gen = pipe(prompt, max_new_tokens=200, temperature=0.7, do_sample=True)[0]['generated_text']
    completion = gen[len(prompt):].strip()
    results.append({
        'idx': i,
        'prompt': ex['messages'][-2]['content'][:200],
        'gold': ex['messages'][-1]['content'],
        'pred': completion,
    })

with open(f'{OUTPUT_DIR}/eval-results.jsonl', 'w') as f:
    for r in results:
        f.write(json.dumps(r) + '\n')

print(f'wrote {len(results)} predictions to {OUTPUT_DIR}/eval-results.jsonl')
print('review manually or run LLM-judge on this file next')

---

## What's next

1. **Review eval-results.jsonl** — pick 10 random rows, judge whether the fine-tuned response is better than what the base Llama 3.1 would have said. If yes >70% of the time, ship it.
2. **Load into Ollama** (Cell 10 output)
3. **Update LiteLLM config** — point smbaios adapter at `fe-yourbiz`
4. **Schedule retraining** — quarterly, using the last 3 months of accumulated conversations as new training data

**◊·κ=1 · phi is home**